# Hermite vs Diffrax Runge--Kutta

This notebook compares `nornax` Hermite solvers against a standard Diffrax RK solver on the same two-body problem.

In [ ]:
import diffrax
import jax
import jax.numpy as jnp

from nornax import AarsethController, solve_adaptive_to_time, total_energy
from nornax.forces import DirectSumGravity

jax.config.update("jax_enable_x64", True)

In [ ]:
force_model = DirectSumGravity()
positions0 = jnp.asarray([[-1.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
velocities0 = jnp.asarray([[0.0, 0.5, 0.0], [0.0, -0.5, 0.0]])
masses = jnp.asarray([1.0, 1.0])

def vector_field(t, y, args):
    positions, velocities = y
    acc = force_model.derivatives(t, positions, velocities, masses, max_order=1).acc
    return velocities, acc

term = diffrax.ODETerm(vector_field)

In [ ]:
hermite8 = solve_adaptive_to_time(
    positions0,
    velocities0,
    masses,
    force_model,
    t_final=2.0,
    order=8,
    controller=AarsethController(eta=0.08, min_dt=1.0e-4, max_dt=5.0e-2),
    atol=1.0e-8,
)

rk = diffrax.diffeqsolve(
    term,
    diffrax.Tsit5(),
    t0=0.0,
    t1=2.0,
    dt0=1.0e-3,
    y0=(positions0, velocities0),
    stepsize_controller=diffrax.PIDController(rtol=1.0e-8, atol=1.0e-8),
    saveat=diffrax.SaveAt(t1=True),
)

rk_positions = rk.ys[0][0]
rk_velocities = rk.ys[1][0]
rk_state = hermite8.final_state._replace(positions=rk_positions, velocities=rk_velocities)

print("Hermite-8 energy:", float(total_energy(hermite8.final_state)))
print("Tsit5 energy:", float(total_energy(rk_state)))
print("Position difference:", float(jnp.linalg.norm(hermite8.final_state.positions - rk_positions)))